# NLS — Risk-Factor Construction Spec (USE4-faithful)

**Purpose.** This notebook is the **source-of-truth spec** for building the **Non-Linear Size (NLS)** style factor exactly as MSCI describes it in *The Barra US Equity Model (USE4) — Empirical Notes, September 2011*. It is **not** a research notebook. The deliverable is a clean, USE4-faithful NLS factor written to parquet, suitable for input to a multi-factor risk model.

**Audience.** You — plus whatever AI assistant you are driving. Treat its output as deeply untrustworthy until audited. Follow the stages linearly. Each stage has:
- ✅ **PDF SPEC** — exact verbatim quote or paraphrase from the USE4 PDF
- ❓ **NOT IN PDF** — something the PDF does not specify; a judgment call you must make
- 💡 **DEFAULT** — a reasonable default for any ❓ item, chosen to match standard Barra practice
- 🧪 **VALIDATE** — sanity checks to run after each stage

**Inputs:** Daily prices from `SHARADAR_SEP.csv`. No fundamentals, no analyst estimates, and **no time-series regression** needed for NLS. The only stock-level input is **market capitalization on signal_date**.

**Deliverable:** A parquet file `nls_use4.parquet` keyed by `(permaticker, signal_date)` containing the standardized NLS exposure for every stock in the coverage universe on every monthly signal date.

**Companion spec:** This notebook mirrors the structure of `02_style_factors/nlb/nlb_spec.ipynb`. NLS is **simpler than NLB** because the input (log market cap) is trivially observable — no rolling regression, no beta estimation, no exponential weights. The orthogonalization / winsorization / standardization machinery downstream is identical to NLB.

---

### Critical reading

The USE4 PDF specifies NLS in **two short places**:
- **Section 3.4 (Style Factors, p.16–17)** — qualitative description
- **Appendix A (p.55)** — formal definition

Both are quoted verbatim in §1 below. Everything else in this spec is either derived from elsewhere in the PDF (standardization convention, regression weights) or marked ❓ NOT IN PDF.

**Bias toward the literal spec.** Where the PDF is ambiguous, follow the conventions established in `nlb_spec.ipynb` so NLB and NLS are constructed consistently. They share a backbone — only the input descriptor differs.

## §1. The USE4 NLS spec — verbatim quotes

### 1a. Section 3.4 (p.16–17) — qualitative description

> *"The Non-Linear Size (NLS) factor captures non-linearities in the payoff to the Size factor across the market-cap spectrum. This factor is based on a single raw descriptor: the cube of the Size exposure. However, since this raw descriptor is highly collinear with the Size factor, it is orthogonalized with respect to Size. This procedure does not affect the fit of the model, but does mitigate the confounding effects of collinearity, while preserving an intuitive meaning for the Size factor. As described by Menchero (2010), the NLS factor roughly captures the risk of a 'barbell portfolio' that is long mid-cap stocks and short small-cap and large-cap stocks."*

### 1b. Appendix A (p.55) — formal definition

> **Non-linear Size**
> 
> *Definition:* `1.0 · NLSIZE`
> 
> *NLSIZE — Cube of Size:* *"First, the standardized Size exposure (i.e., log of market cap) is cubed. The resulting factor is then orthogonalized to the Size factor on a regression-weighted basis. Finally, the factor is winsorized and standardized."*

### 1c. Appendix A (p.51) — Size definition (input to NLS)

> **Size**
> 
> *Definition:* `1.0 · LNCAP`
> 
> *LNCAP — Log of market cap:* *"Given by the logarithm of the total market capitalization of the firm."*

### 1d. Section 3.4 (p.15) — standardization convention (applies to every style factor including NLS)

> *"In order to facilitate comparison across style factors, the factors are standardized to have a cap-weighted mean of 0 and an equal-weighted standard deviation of 1. The cap-weighted estimation universe, therefore, is style neutral."*

### 1e. Section 3.1 (p.7) — Estimation Universe

> *"The USE4 estimation universe utilizes the MSCI USA Investable Markets Index (USA IMI), which aims to reflect the full breadth of investment opportunities within the US market by targeting 99 percent of the float-adjusted market capitalization. ... Moreover, liquidity screening rules are applied to ensure that only investable stocks that meet the index methodological requirements are included for index membership."*

### 1f. Appendix B (p.56) — regression weights are sqrt(market cap)

> *"where v_n is the regression weight of stock n (proportional to square-root of market capitalization)."*

### 1g. Table 4.2 caption (p.26) — sqrt-mcap weights confirmed for style factor regressions

> *"The factor stability coefficient and Variance Inflation Factor were computed on monthly data using square root of market-cap weighting."*

---

**That is all the PDF says about NLS construction.** Every additional decision required to implement this (universe rules, winsorization percentiles, treatment of edge cases) is ❓ **NOT IN PDF** and is flagged as such below.

**Note on the barbell interpretation.** The PDF says NLS is long mid-caps, short small AND large caps. This is the qualitative shape; in practice the residual from regressing cube-of-z on z (with intercept) produces a function `z³ − β·z` whose sign flips at `|z| ≈ √(E[z⁴]/E[z²])`. Don't over-engineer the interpretation — just construct the residual per spec and let the cross-sectional pattern emerge naturally.

## §2. End-to-end pipeline

```
┌────────────────────────────────────────────────────────────────────┐
│  UPSTREAM:  estu_build.ipynb   →  estu_monthly.parquet             │
│             daily_build.ipynb  →  daily_returns.parquet (+ maps)   │
│             size_build.ipynb   →  size_use4.parquet                │
├────────────────────────────────────────────────────────────────────┤
│  STAGE 0:  Setup, parameters                                       │
│  STAGE 1:  Load shared daily-returns artifact                      │
│  STAGE 2:  Load shared ESTU                                        │
│  STAGE 3:  Size input from size_use4.parquet (lncap, size_score)   │
│  STAGE 4:  size_z := size_score (standardized upstream)            │
│  STAGE 5:  Cube → size_z³                                          │
│  STAGE 6:  Orthogonalize to Size (√mcap WLS, ESTU fit)             │
│  STAGE 7:  Winsorize (1%/99% ESTU quantiles)                       │
│  STAGE 8:  Standardize (CW mean = 0, EW std = 1)                   │
│  STAGE 9:  Save deliverable                                        │
│  STAGE 10: Validate                                                │
└────────────────────────────────────────────────────────────────────┘
```

**PIT discipline:** for any signal_date `t`, only data dated ≤ `t` is used.

**One-sweep run:** `your end-to-end runner` executes the whole
pipeline in dependency order (sequential, one kernel at a time).

## §3. Output schema — what the worker delivers

**Raw descriptor column:** `nls_raw`
**Standardized score column:** `nls_score`

**File:** `data/out/nls_use4.parquet`

**Compression:** zstd, statistics=True

**Schema:**

| Column | Type | Description |
|---|---|---|
| `permaticker` | Int64 | Sharadar permanent ticker ID |
| `signal_date` | Date | End-of-month rebalance date (last trading day of month) |
| `in_estu` | Bool | Whether this stock was in the estimation universe on this date |
| `mcap` | Float64 | Total market cap on signal_date |
| `log_mcap` | Float64 | `ln(mcap)` — the raw Size descriptor |
| `size_z` | Float64 | Standardized Size (cap-weighted mean 0, EW std 1) |
| `size_z_cubed` | Float64 | `size_z ^ 3` |
| `nls_raw` | Float64 | Residual after orthogonalization to Size |
| `nls_score` | Float64 | **Final NLS exposure** — winsorized, standardized (the actual deliverable) |

**Coverage rules:**
- Compute `log_mcap`, `size_z`, `size_z_cubed`, `nls_raw`, `nls_score` for **every stock with valid mcap > 0**, not just ESTU members
- Standardization and orthogonalization use **ESTU members only** to compute the moments (means, stds, regression slopes)
- Non-ESTU stocks get the *same* standardization parameters applied so the final factor is comparable across the coverage universe

This matches USE4's distinction between *coverage universe* (everyone gets a forecast) and *estimation universe* (subset used to estimate model parameters).

## §4. STAGE 0 — Setup, imports, paths

Standard imports. Use polars, not pandas, throughout (project convention).

Standard imports. Use polars, not pandas, throughout (project convention).

```python
# ── Factor type (read by /build-factor to select boilerplate template) ────────
FACTOR_TYPE = "time_series"   # "time_series" or "fundamental"

# ── Parameters ────────────────────────────────────────────────────────────────
# (Factor-specific parameters are defined in the Stage 0 code cell below.)
```

```python
# ───── Parameters ─────────────────────────────────────────────────────────────
FACTOR_TYPE = "time_series"

# 💡 DEFAULT (NOT IN PDF) — winsorization percentiles (PDF says "winsorized" only)
WINSOR_LO = 0.01
WINSOR_HI = 0.99

# 💡 DEFAULT (NOT IN PDF) — calendar start (match the other factors)
CALENDAR_START = date(1999, 1, 1)
```

## §5. STAGE 1 — Load the shared daily panel

By now you have built this plumbing inline at least once (Beta). This factor
consumes the **shared daily panel** instead — the refactor step specified in
`01.5_daily/daily_spec.ipynb`:

| Variable | File (in `data/out/`) | Contents |
|---|---|---|
| `daily` | `daily_returns.parquet` | `permaticker, date, ret, mcap_lag, mkt_ret` — excess log returns; `mkt_ret` is the ESTU cap-weighted excess benchmark |
| `prices` | `ticker_permaticker.parquet` | ticker → permaticker map (spot-check lookups) |

**Build order:** `estu_build.ipynb` → `daily_build.ipynb` → this factor.

🧪 **VALIDATE:** performed once in the daily-panel build — benchmark correlation
> 0.90, crisis-vol sanity, load-path equivalence, missing-benchmark dates < 30.

## §6. STAGE 2 — Load the shared ESTU (`estu_monthly.parquet`)

✅ **PDF SPEC:** USE4 ESTU = MSCI USA Investable Markets Index (USA IMI) — ~99% of
US float-adjusted market cap, liquidity- and stability-screened, ~2,500–3,000 names.

💡 **DEFAULT:** load the shared `data/out/estu_monthly.parquet` built by
`estu_build.ipynb` (spec: `01_estu/estu_spec.ipynb`): top ~3,000
domestic common stocks on NYSE/NASDAQ/NYSEMKT, ATVR liquidity screen
(add ≥ 20%, retain ≥ 10%), 3,000/3,500 hysteresis buffer. **Every factor must use
this same ESTU** — factor-specific universes would break the multi-factor
cross-sectional regression.

🧪 **VALIDATE:**
- 330 monthly signal dates (1999-01-29 →); ESTU size mean ≈ 2,500, range ≈ 1,800–3,050
- Mega-caps (AAPL, MSFT, GOOGL, JPM) present every month they were public

## §7. STAGE 3 — Size input from the SIZE factor deliverable

✅ **PDF SPEC (Appendix A, p.51):** `LNCAP = ln(total market capitalization of the
firm)`; Size = `1.0 · LNCAP`.

💡 **DEFAULT (2026-06-10):** `nls_build.ipynb` reads
`data/out/size_use4.parquet` (spec: `size/size_spec.ipynb`):
`log_mcap := lncap` (post-trim) and `size_z := size_score`. NLS is thereby
orthogonalized to the *same* Size exposure that enters the risk model.
Rebuild order: `size_build.ipynb` → `nls_build.ipynb`.

🧪 **VALIDATE:**
- `log_mcap` median ≈ 19.9 (Sharadar mcap in dollars; ln(4×10⁸) ≈ 19.9)
- `size_z` has CW mean ≈ 0 and EW std ≈ 1 within ESTU (by construction upstream)

## §8. STAGE 4 — Standardized Size (handled upstream)

✅ **PDF SPEC (Section 3.4, p.15):** *"factors are standardized to have a
cap-weighted mean of 0 and an equal-weighted standard deviation of 1."* The cube
is applied to the **standardized** Size exposure.

💡 **DEFAULT:** `size_z := size_score` arrives pre-standardized from the SIZE build
(ESTU moments, applied to all stocks). No computation in this notebook.

## §9. STAGE 5 — Cube

✅ **PDF SPEC:** *"First, the standardized Size exposure (i.e., log of market cap) is cubed."*

Trivially:

```
size_z_cubed_{i,t} = size_z_{i,t} ^ 3
```

Apply per row. No cross-sectional aggregation needed at this step.

**Why cube?** A z-score of 2 (very large cap relative to mean) becomes 8 in cubed space. A z-score of −2 (very small cap) becomes −8. A z-score of 0.5 becomes 0.125. The cube amplifies tails while preserving sign. This is what makes the residual after orthogonalization capture the "barbell" — the deviation from linear-in-log-mcap behavior at the size extremes.

**This step is identical to NLB Stage 6.** The cube of a standardized variable is the cube of a standardized variable — the operation doesn't care what the underlying variable is.

🧪 **VALIDATE:**
- For a stock with `size_z = 0`, `size_z_cubed = 0`
- For `size_z = 1.0`, `size_z_cubed = 1.0`
- For `size_z = 2.0`, `size_z_cubed = 8.0`
- Distribution is much more skewed/extreme than `size_z`

## §10. STAGE 6 — Orthogonalize to Size (regression-weighted, sqrt-mcap)

**This is the most important and most error-prone step.** Identical machinery to NLB Stage 7 — only the regressor (`log_mcap` here vs. `linear_beta` for NLB) changes.

✅ **PDF SPEC (Appendix A, p.55):** *"The resulting factor is then orthogonalized to the Size factor on a regression-weighted basis."*

✅ **PDF SPEC (Appendix B, p.56 + Table 4.2 caption, p.26):** "Regression-weighted" = **sqrt(market cap) weighted**. This is the standard CSR weighting in USE4.

**Procedure (per signal_date `t`, using ESTU members to fit, applied to everyone):**

1. Restrict to ESTU members with valid `log_mcap` and valid `size_z_cubed`
2. Define regression weights: `v_i = sqrt(mcap_i)`
3. Run **weighted OLS**: regress `size_z_cubed_i` on `log_mcap_i` (with intercept), with weights `v_i`
4. Save the coefficients `(α_t, β1_t)`
5. Compute residual for **every stock** (ESTU and non-ESTU) on date `t`:

```
nls_raw_{i,t} = size_z_cubed_{i,t} − α_t − β1_t · log_mcap_{i,t}
```

**Weighted OLS formulas (with intercept):**

Define centered values using sqrt-mcap weights:
```
V  = Σ v_i
x̄ = Σ(v_i · x_i) / V                # weighted mean of log_mcap
ȳ = Σ(v_i · y_i) / V                # weighted mean of size_z_cubed
x_c = x − x̄
y_c = y − ȳ

β1 = Σ(v_i · x_c · y_c) / Σ(v_i · x_c²)
α  = ȳ − β1 · x̄
```

Then residual:
```
nls_raw_i = y_i − α − β1 · x_i
         = y_c_i − β1 · x_c_i
```

**A subtle choice: regress on `log_mcap` or on `size_z`?** Mathematically the residuals are identical, since `size_z` is just a linear transformation of `log_mcap` (different intercept and slope, same residual after centering). For clarity and to mirror the PDF wording ("orthogonalized to the Size factor"), regress against `log_mcap` — that's the literal Size descriptor per Appendix A. The choice has zero numerical impact.

**Critical bug to avoid (same as NLB):** Don't use `mcap` (linear) weights instead of `sqrt(mcap)`. USE4 spec is **sqrt(mcap)**. Linear mcap would pin mega-cap residuals to zero too aggressively.

❓ **NOT IN PDF — pairwise vs joint orthogonalization.** Appendix A says "orthogonalize to the Size factor" — pairwise. Same ambiguity as NLB. 💡 **DEFAULT:** Pairwise (Size only), per Appendix A's literal reading.

🧪 **VALIDATE — the critical check:**

After computing `nls_raw`, verify the sqrt-mcap-weighted orthogonality condition holds on ESTU:

```
Σ_{i ∈ ESTU} (sqrt(mcap_i) · log_mcap_i · nls_raw_i) / Σ_{i ∈ ESTU} sqrt(mcap_i)  ≈ 0
```

By construction this should be **machine-zero**. If it's not, the orthogonalization is broken.

Also: unweighted Pearson correlation between `nls_raw` and `log_mcap` *will not be zero* and that's fine. The relevant orthogonality is the sqrt-mcap-weighted one above.

**Read this twice.** This is the place where most implementations get USE4 wrong.

## §11. STAGE 7 — Winsorize

✅ **PDF SPEC:** *"the factor is winsorized"*

❓ **NOT IN PDF — winsorization percentiles.** Same ambiguity as NLB.

💡 **DEFAULT:** **1%/99% percentiles**, computed **per signal_date** on ESTU only, applied **per signal_date** to all rows. Same choice as NLB.

**Formula:**
```
lo_t = quantile(nls_raw_{·,t} | ESTU, 0.01)
hi_t = quantile(nls_raw_{·,t} | ESTU, 0.99)

nls_winsor_{i,t} = clip(nls_raw_{i,t}, lo_t, hi_t)
```

Quantiles computed on ESTU only, clip applied to everyone.

🧪 **VALIDATE:**
- Distribution should look approximately bell-shaped after winsorization
- ~1% of ESTU rows should be at exactly `lo_t`, ~1% at `hi_t`
- Non-ESTU stocks may show higher clip rates if their distribution differs from ESTU

## §12. STAGE 8 — Final standardize (cap-weighted mean=0, EW std=1)

✅ **PDF SPEC:** *"Finally, the factor is winsorized and standardized."*

Same procedure as §8 (Stage 4), applied to `nls_winsor` instead of `log_mcap`:

1. Restrict to ESTU members with valid `nls_winsor`
2. Compute cap-weighted mean `μ_t` using `mcap` as weights
3. Compute equal-weighted std `σ_t` of `(nls_winsor − μ_t)`
4. Apply `(nls_winsor − μ_t) / σ_t` to **every** row (ESTU and non-ESTU)
5. The result is `nls_score` — the final deliverable

🧪 **VALIDATE:**
- `cap_weighted_mean(nls_score | ESTU) ≈ 0`
- `equal_weighted_std(nls_score | ESTU) ≈ 1`
- Range typically [−3, +3] for ESTU stocks (since we winsorized at 1/99 before standardizing)
- Distribution: roughly bell-shaped

**Order matters.** Winsorize → standardize. Don't standardize → winsorize → standardize. The PDF order is: winsorize, then standardize.

After this stage, `nls_score` is **the final NLS exposure** ready for use in a risk model.

## §13. STAGE 9 — Save deliverable

Write the final panel to `data/out/nls_use4.parquet` with the schema from §3.

Compression: zstd. Statistics: True.

Include all the intermediate columns (`log_mcap`, `size_z`, `size_z_cubed`, `nls_raw`) for audit purposes — they're small and make debugging downstream issues much easier.

## §14. STAGE 10 — Validation

Run these checks **on the saved deliverable**, not on intermediate state. The deliverable is what gets consumed downstream, so the deliverable is what must pass validation.

### Required checks (all must pass before signing off)

**1. Sqrt-mcap-weighted orthogonality to Size (the key USE4 property):**
```
For each signal_date t, computed over ESTU:
    r_sqrt_t = Σ(sqrt(mcap_i) * log_mcap_i * nls_score_i) / Σ(sqrt(mcap_i))
Should be < 1e-10 in absolute value, every date.
```

**2. Cap-weighted mean of nls_score is zero:**
```
For each signal_date t, computed over ESTU:
    Σ(mcap_i * nls_score_i) / Σ(mcap_i) ≈ 0
```

**3. Equal-weighted std of nls_score is one:**
```
For each signal_date t, computed over ESTU:
    std(nls_score_i for i in ESTU) ≈ 1.0
```

**4. Coverage stability:**
```
Per signal_date, count of stocks with non-null nls_score should be stable over time.
Plot it. Sudden drops indicate a bug.
```

**5. Factor stability (month-over-month rank correlation):**
```
For consecutive signal_dates t and t+1, compute Spearman correlation
of nls_score for stocks present in both dates.
USE4 reports NLS stability of ~0.98-0.99 (Table 4.2). Target > 0.95.
NLS should be VERY stable since log(mcap) barely changes month-to-month.
```

**6. Equivalence vs reference implementation:**
```
Spot check at least one signal_date with a numpy-loop reference
implementation of the pipeline. Confirm max abs diff < 1e-10.
```

### Optional diagnostic checks

- Cross-sectional Pearson correlation (unweighted) with `log_mcap`. Won't be zero — fine for USE4. Report it for reference.
- Histogram of `nls_score` per signal_date — should be roughly bell-shaped, centered ~0, range mostly [−3, +3]
- Check the barbell shape: scatter `nls_score` vs `size_z` (or `log_mcap`). The shape should resemble a "smile" or "frown" — non-monotonic, with extrema at intermediate values of size_z. This visualizes the non-linear deviation from a linear size effect.

---

**Shared validation conventions (all factors, 2026-06-10):**
- **Coverage (check 3)** is evaluated on **completed months only** — the final
  signal date is the in-progress month (freshest data can lag the Sharadar
  refresh) and is excluded from the floor.
- **Fraction of scores in [−3, +3]** is reported for the full universe *and* for
  ESTU; the ESTU figure is the operationally relevant one (non-ESTU micro-caps
  pull the all-universe tail).
- Common check machinery lives in `common/`, your shared utilities.

## §15. Master list of ❓ NOT-IN-PDF decisions

NLS has fewer ambiguous parameters than NLB because there's no beta estimation. Every place this spec made a judgment call not specified in USE4:

| # | Decision | Default chosen | Alternative | Where to revisit |
|---|---|---|---|---|
| 1 | ESTU construction (MSCI USA IMI is proprietary) | Shared `estu_monthly`: top 3000, ATVR screen, 3000/3500 buffer | Top 2500, different liquidity threshold, MSCI-style buffer | If ESTU churn > 5%/month |
| 2 | Winsorization percentiles | 1% / 99% | 2.5%/97.5%, ±3σ | If validation shows fat tails after winsorization |
| 3 | Pairwise vs joint orthogonalization | Pairwise (Size only) per Appendix A literal reading | Joint against [Size, Beta, …] | When building the multi-factor risk model — orthogonalize jointly in the full CSR |
| 4 | ESTU buffer rule for stability | None (re-rank monthly) | Implement addition/deletion thresholds | If month-over-month exposure stability < 0.95 |
| 5 | Float-adjusted mcap | Total mcap (Sharadar doesn't have float) | Procure float data | If running for institutional-grade comparison to MSCI |
| 6 | Sharadar `marketcap` unit (dollars vs millions) | Dollars (ln values ≈ 18–23); documented in code and audits | — | Document the unit in the code; downstream consumers may care |
| 7 | Calendar start | 1999-01-01 (match NLB) | Earlier if data available | Probably never — match NLB for consistency |
| 8 | Regress cube on `log_mcap` or on `size_z` | Either — residuals are identical | — | Pure stylistic choice |

Items NOT on this list (that appear on the NLB list) — and why:
- Risk-free rate: irrelevant for NLS (no excess returns needed)
- Beta window length, half-life, MIN_OBS: irrelevant (NLS has no time-series regression)
- Daily vs monthly ESTU for benchmark: irrelevant (NLS doesn't need a daily market benchmark)

NLS is meaningfully simpler than NLB.

## §16. Common pitfalls — read this before coding

**Pitfall 1: Mixing weighting schemes.**
- Standardization mean: **cap-weighted** (mcap)
- Standardization std: **equal-weighted** (no weights)
- Orthogonalization regression: **sqrt(mcap)-weighted**

Same as NLB. Three different weighting schemes in three different places. Don't homogenize them.

**Pitfall 2: Standardizing on the wrong universe.**
- Moments (mean, std, regression coefficients) are computed using **ESTU only**
- The standardization/orthogonalization is **applied to every stock** in the coverage universe

**Pitfall 3: Confusing sqrt-mcap orthogonality with unweighted Pearson.**
- USE4-correct: sqrt-mcap-weighted residual is orthogonal to log_mcap (Σ = 0)
- USE4 says nothing about unweighted Pearson r(nls_raw, log_mcap) being zero
- Don't "fix" the implementation to make unweighted Pearson zero — that breaks the USE4 property

**Pitfall 4: Forgetting the intercept in the orthogonalization regression.**
- `log_mcap` averages around 7 (if mcap in millions) or 20 (if mcap in dollars) — NOT zero
- Without an intercept, the slope is biased
- The centering approach (subtracting weighted means from both x and y) implicitly handles this

**Pitfall 5: Using log of a non-positive number.**
- Filter `mcap > 0` *before* taking `log`
- A stock with `mcap = 0` or null should be dropped from the panel, not silently produce `-inf` or `null`

**Pitfall 6: Cubing before standardizing.**
- The PDF says "the **standardized** Size exposure is cubed"
- If you cube `log_mcap` directly (without standardizing first) you get values around 7³ ≈ 343, not the small numbers around z=0 you want
- Order: standardize → cube → orthogonalize → winsorize → standardize

**Pitfall 7: PIT violations through joins.**
- For NLS this is less of a worry because there's no rolling window — just need point-in-time mcap
- But still: never use `marketcap` from a date after `signal_date`
- If using join_asof, use `strategy="backward"` to ensure no look-ahead

**Pitfall 8: Float-adjusted vs total market cap.**
- Sharadar `marketcap` is total
- MSCI uses float-adjusted
- For US large-caps the difference is small (5–20%) but it shifts log_mcap slightly
- Document the choice; don't switch midstream

**Pitfall 9: Treating NLS as a return signal.**
- NLS is a **risk** factor. USE4 reports its return Sharpe at roughly 0 (Table 4.2)
- A backtest of Q5−Q1 NLS spread is not expected to produce alpha
- Don't be surprised if the L/S spread is flat or negative — it's a feature, not a bug

**Pitfall 10: Forgetting that NLS exposures barely change month-to-month.**
- log(mcap) only changes by ~mcap return per month, which is typically a few percent
- size_z therefore moves slowly
- Cube and orthogonalize preserve this slowness
- USE4 Table 4.2 reports NLS factor stability coefficient of 0.98–0.99 (very high)
- If your stability drops much below 0.95, something is wrong — likely a bug in the standardization or ESTU step

## §17. Final summary — what "done" looks like

**The deliverable is one file:** `data/out/nls_use4.parquet`

**Acceptance criteria:**

1. ✅ Schema matches §3
2. ✅ All 6 required validation checks in §14 pass within tolerance
3. ✅ ESTU has ~2500–3000 names per date, stable over time (from your ESTU build, step 01)
4. ✅ Sqrt-mcap-weighted orthogonality to Size (log_mcap) is machine-zero for every date
5. ✅ Cap-weighted mean of nls_score is machine-zero for every date in ESTU
6. ✅ Equal-weighted std of nls_score is 1.0 (to 6 decimals) for every date in ESTU
7. ✅ Coverage of nls_score across coverage universe is ≥ 4000 stocks per date in recent years
8. ✅ Month-over-month rank stability ρ > 0.95 for ESTU members (NLS is very stable)
9. ✅ Numerical equivalence between the polars build and a numpy reference, max abs diff < 1e-10
10. ✅ All ❓ NOT-IN-PDF decisions in §15 are documented somewhere in the code (comment or markdown)

If all 10 are satisfied, NLS v1 is ready to plug into the risk model.

---

**What NLS reuses:**
- the monthly signal calendar and standardization helpers from your earlier builds (or your `common/` helpers)
- the sqrt(mcap)-WLS orthogonalization you wrote for NLB (`nlb_build.ipynb`) — swap in the `log_mcap` descriptor for `linear_beta` and rename variables
- the cross-sectional correlation check from your NLB build — adapt it for NLS
- NLS loads no prices: it reads `size_score` and `log_mcap` from `size_use4.parquet` (your `size_build.ipynb`)

**What this notebook produces vs NLB:**
- Same backbone (standardize, cube, orthogonalize, winsorize, standardize)
- Different input descriptor (log_mcap instead of estimated linear beta)
- Different barbell interpretation (mid-caps long, extremes short — vs NLB which is avg-beta long, extreme-beta short)
- Different stability profile (NLS is MORE stable than NLB because log_mcap is a much smoother input than rolling beta)

**Once both NLS and NLB are built and pass validation, the next step is to:**
1. Run cross-factor diagnostics (correlation between `nls_score` and `nlb_score` should be modest, not 1.0)
2. Verify both factors fit cleanly into the multi-factor cross-sectional regression
3. Confirm NLS against its chapter in `textbook/` (the Measured boxes are the reference numbers); the factor earns its slot when it enters the cross-sectional regression (step `05_csr`)

**Pro tip:** If you find yourself writing very similar code for NLS and NLB, factor the shared machinery (sqrt-mcap WLS orthogonalization, cap-weighted-mean / EW-std standardization, winsorization) into reusable helpers in a shared `common/` module. The same pattern will apply to Residual Volatility (orthogonalized to Beta) and any future cubed-descriptor factor.